# SCBE Pivot Training v2 — Controlled Conversation Engine

**Goal:** Generate high-quality SFT and DPO training data with measured pivot quality.

**What's different from v1:**
- Pivots have a **control metric** (relevance, growth, novelty, coherence)
- Growth parameters are tracked and expanded
- Conversations build from first principles
- ChoiceScript-style branching generates DPO preference pairs

**Corpus pillars (all IP-safe):**
1. The Six Tongues Protocol (Issac's novel)
2. Everweave game lore (12,596 paragraphs)
3. SCBE system knowledge
4. Math from first principles
5. ChoiceScript decision trees
6. Open source literature

In [ ]:
# ============================================================
# Cell 1: Install deps + mount Drive
# ============================================================
!pip install -q datasets huggingface_hub numpy

from google.colab import drive
drive.mount('/content/drive')

import json, random, hashlib, math, time
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Optional, Tuple, Set
from pathlib import Path
from collections import defaultdict

print('Ready.')

In [ ]:
# ============================================================
# Cell 2: Pivot Quality Metric
# ============================================================

@dataclass
class PivotScore:
    """Control metric for pivot quality. Not random — measured."""
    relevance: float    # 0-1: did pivot connect to prior context?
    growth: float       # 0-1: did we learn something new?
    novelty: float      # 0-1: is this a path not yet taken?
    coherence: float    # 0-1: does conversation still make sense?

    @property
    def quality(self) -> float:
        """Weighted composite. Coherence matters most."""
        return (
            0.20 * self.relevance +
            0.25 * self.growth +
            0.20 * self.novelty +
            0.35 * self.coherence
        )

    @property
    def is_good(self) -> bool:
        return self.quality >= 0.5

    def to_dict(self):
        d = asdict(self)
        d['quality'] = self.quality
        d['is_good'] = self.is_good
        return d


# Quick test
good_pivot = PivotScore(relevance=0.8, growth=0.7, novelty=0.6, coherence=0.9)
bad_pivot = PivotScore(relevance=0.1, growth=0.1, novelty=0.9, coherence=0.1)
print(f'Good pivot: {good_pivot.quality:.3f} ({"PASS" if good_pivot.is_good else "FAIL"})')
print(f'Bad pivot:  {bad_pivot.quality:.3f} ({"PASS" if bad_pivot.is_good else "FAIL"})')

In [ ]:
# ============================================================
# Cell 3: Topic Graph with Sacred Tongue Affinity
# ============================================================

SACRED_TONGUES = {
    'KO': {'name': "Kor'aelin",  'domain': 'Authority / Control',     'neurotransmitter': 'Dopamine'},
    'AV': {'name': 'Avali',       'domain': 'Transport / Messaging',   'neurotransmitter': 'Acetylcholine'},
    'RU': {'name': 'Runethic',    'domain': 'Policy / Constraints',    'neurotransmitter': 'Serotonin'},
    'CA': {'name': 'Cassisivadan','domain': 'Compute / Encryption',    'neurotransmitter': 'Glutamate'},
    'UM': {'name': 'Umbroth',     'domain': 'Security / Secrets',      'neurotransmitter': 'GABA'},
    'DR': {'name': 'Draumric',    'domain': 'Schema / Authentication', 'neurotransmitter': 'Cortisol'},
}

@dataclass
class Topic:
    id: str
    name: str
    tongue: str  # KO/AV/RU/CA/UM/DR
    depth: int   # 0=surface, 1=intermediate, 2=deep
    keywords: List[str]
    connections: List[str]  # topic IDs this can pivot to
    questions: List[str]    # instruction prompts
    responses: List[str]    # correct response templates
    bad_responses: List[str] = field(default_factory=list)  # for DPO


# Build a topic graph from first principles
TOPICS = {
    # === MATH (first principles) ===
    'math_binary': Topic(
        id='math_binary', name='Binary Numbers', tongue='CA', depth=0,
        keywords=['binary', 'bits', 'base 2', '0 and 1'],
        connections=['math_arithmetic', 'math_logic', 'crypto_basics'],
        questions=[
            'What is binary and why do computers use it?',
            'Convert the number 42 to binary.',
            'Why is base 2 fundamental to computation?',
        ],
        responses=[
            'Binary is base-2 numbering using only 0 and 1. Computers use it because transistors have two states: on (1) and off (0). Every computation, from adding numbers to rendering graphics, reduces to operations on binary digits.',
            '42 in binary is 101010. To convert: 42 = 32+8+2 = 2^5 + 2^3 + 2^1, so bits 5,3,1 are set.',
            'Base 2 maps directly to physical switches. Any system with two distinguishable states — voltage high/low, magnetized/not, photon present/absent — can represent binary. This universality is why binary underpins all digital systems.',
        ],
        bad_responses=[
            'Binary is just 1s and 0s, nothing special.',
            '42 in binary is 110010. (Wrong answer for DPO rejection.)',
        ],
    ),
    'math_arithmetic': Topic(
        id='math_arithmetic', name='Arithmetic', tongue='CA', depth=0,
        keywords=['addition', 'multiplication', 'division', 'modular'],
        connections=['math_binary', 'math_algebra', 'math_logic'],
        questions=[
            'Explain modular arithmetic and why it matters for cryptography.',
            'What is the golden ratio and where does it appear in nature?',
        ],
        responses=[
            'Modular arithmetic is clock math — numbers wrap around at a modulus. 7 mod 3 = 1 because 7 = 2*3 + 1. It matters for cryptography because operations in a finite field are easy to compute forward but hard to reverse, which is the basis of RSA and elliptic curve cryptography.',
            'The golden ratio phi = (1+sqrt(5))/2 ≈ 1.618. It appears in spiral galaxies, sunflower seed arrangements, nautilus shells, and the Fibonacci sequence. In SCBE, phi governs Sacred Tongue weight scaling: KO=1.00, AV=phi, RU=phi^2, CA=phi^3, UM=phi^4, DR=phi^5.',
        ],
        bad_responses=[
            'Modular arithmetic is when you use a calculator.',
        ],
    ),
    'math_fft': Topic(
        id='math_fft', name='Fourier Transform', tongue='CA', depth=2,
        keywords=['FFT', 'frequency', 'spectrum', 'harmonic', 'sine wave'],
        connections=['math_arithmetic', 'audio_basics', 'scbe_spectral'],
        questions=[
            'What does the Fourier Transform do in simple terms?',
            'Why is FFT O(N log N) instead of O(N^2)?',
            'How does spectral analysis connect to the SCBE 14-layer pipeline?',
        ],
        responses=[
            'The Fourier Transform decomposes a signal into its component frequencies. Like a prism splitting white light into a rainbow, FFT splits a sound wave into the individual sine waves that make it up. Each frequency has an amplitude (how loud) and a phase (when it starts).',
            'The Cooley-Tukey FFT algorithm exploits symmetry in the DFT matrix. By splitting the N-point DFT into two N/2-point DFTs recursively, it reduces N^2 multiplications to N*log2(N). For N=1024, that is 10240 vs 1048576 — a 100x speedup.',
            'SCBE layers 9-10 use FFT for spectral coherence analysis. Layer 9 decomposes the context signal into frequency components. Layer 10 checks spin coherence — whether the phase relationships between harmonics are consistent. Inconsistent phases indicate adversarial manipulation, like a forged voice with mismatched overtones.',
        ],
        bad_responses=[
            'FFT is a fast algorithm. It makes things faster.',
        ],
    ),

    # === SCBE SYSTEM ===
    'scbe_poincare': Topic(
        id='scbe_poincare', name='Poincare Ball Model', tongue='UM', depth=1,
        keywords=['hyperbolic', 'poincare', 'trust ring', 'curvature'],
        connections=['math_fft', 'scbe_spectral', 'scbe_governance', 'scbe_tongues'],
        questions=[
            'Why does SCBE use hyperbolic geometry instead of Euclidean?',
            'What are the four trust rings in the Poincare ball?',
        ],
        responses=[
            'Hyperbolic space grows exponentially with distance from the center. In the Poincare ball, moving from radius 0.3 to 0.9 covers exponentially more "space" than 0 to 0.3. This means adversarial drift costs exponentially more the further it goes — attacks become computationally infeasible, not just expensive.',
            'CORE (r<0.3, 5ms latency): highest trust, critical ops. INNER (0.3-0.7, 30ms): standard authenticated. OUTER (0.7-0.9, 200ms): partially verified. WALL (r>=0.9): absolute deny. The exponential cost scaling of hyperbolic distance makes the WALL mathematically impenetrable.',
        ],
        bad_responses=[
            'SCBE uses hyperbolic geometry because it sounds cool.',
        ],
    ),
    'scbe_tongues': Topic(
        id='scbe_tongues', name='Sacred Tongues', tongue='KO', depth=1,
        keywords=['sacred', 'tongues', 'KO', 'AV', 'RU', 'CA', 'UM', 'DR', 'langues'],
        connections=['scbe_poincare', 'scbe_governance', 'lore_characters'],
        questions=[
            'Name the six Sacred Tongues and their neurotransmitter mappings.',
            'Why are there exactly six tongues?',
            'How do tongue weights scale and why?',
        ],
        responses=[
            "KO (Kor'aelin) = Dopamine (reward/authority). AV (Avali) = Acetylcholine (attention/transport). RU (Runethic) = Serotonin (mood/policy). CA (Cassisivadan) = Glutamate (excitation/compute). UM (Umbroth) = GABA (inhibition/security). DR (Draumric) = Cortisol (stress/authentication). Together they form a neurochemically-inspired control plane balancing excitation and inhibition.",
            'Six maps to three excitatory-inhibitory pairs: KO-UM (command-guard), AV-RU (transport-constraint), CA-DR (compute-structure). This balance mirrors biological neural regulation. Six also maps to the faces of a cube, vertices of an octahedron, and strings of a guitar.',
            'Weights scale by phi (golden ratio): KO=1.00, AV=1.62, RU=2.62, CA=4.24, UM=6.85, DR=11.09. Higher tongues are exponentially more expensive to activate, creating natural scarcity for high-privilege operations. DR (authentication/structure) costs 11x more than KO (intent/flow).',
        ],
        bad_responses=[
            'There are six tongues because six is a nice number.',
        ],
    ),

    # === LORE ===
    'lore_characters': Topic(
        id='lore_characters', name='Spiralverse Characters', tongue='DR', depth=1,
        keywords=['Marcus', 'Polly', 'Senna', 'Kael', 'Bram', 'Sera'],
        connections=['scbe_tongues', 'scbe_governance', 'lore_world'],
        questions=[
            'Who is Marcus Chen and what is his role in the Spiralverse?',
            'Describe Polly and her relationship to the Archive.',
            'What happened to Kael Thorne and why does it matter?',
        ],
        responses=[
            'Marcus Chen is a security engineer from Earth who gets pulled into Aethermoor. He becomes the first person in centuries to interact with all six Sacred Tongues simultaneously. His Earth-trained pattern recognition lets him see vulnerabilities in Aethermoor\'s governance that natives are blind to — making him both invaluable and dangerous.',
            'Polly (Polivara) is the Fifth Circle Keeper of the Archives — a raven who can shift to humanoid form. She is KO-aligned (authority/control) and serves as Marcus\'s primary guide. She is ancient, sarcastic, deeply loyal, and carries the weight of knowing how close Aethermoor\'s governance came to collapse.',
            'Kael Thorne gamed the trust model and was erased 300 years ago. But his pattern persists in the infrastructure like a ghost — 17 copies of his signature were discovered in Chapter 13, requiring a 72-drone swarm purge. He represents the fundamental question: can governance survive someone who understands it completely?',
        ],
        bad_responses=[
            'Marcus is the main character. He does stuff.',
            'Polly is a bird.',
        ],
    ),
    'lore_world': Topic(
        id='lore_world', name='Aethermoor World', tongue='DR', depth=0,
        keywords=['Aethermoor', 'Spiralverse', 'world', 'circles', 'governance'],
        connections=['lore_characters', 'scbe_tongues', 'scbe_governance'],
        questions=[
            'What is Aethermoor?',
            'How does the Circle system work?',
        ],
        responses=[
            'Aethermoor is a world where magic operates through governance protocols. There is no wild magic — every spell, transport, and transformation runs through the Sacred Tongue infrastructure. The Circles are trust tiers (First through Seventh) that determine what operations an individual can perform. Breaking Circle constraints triggers the harmonic wall.',
            'The Circle system mirrors the Poincare trust rings. First Circle is outermost (limited access). Seventh Circle is innermost (near-absolute authority). Advancement requires demonstrated mastery of progressively more Sacred Tongues. Each Circle unlocks new governance privileges but also new responsibilities and constraints.',
        ],
        bad_responses=[
            'Aethermoor is a fantasy world with magic.',
        ],
    ),

    # === AUDIO (new frontier) ===
    'audio_basics': Topic(
        id='audio_basics', name='Digital Audio Fundamentals', tongue='AV', depth=0,
        keywords=['audio', 'sample rate', 'waveform', 'Hz', 'amplitude'],
        connections=['math_fft', 'scbe_spectral', 'audio_voice'],
        questions=[
            'What is digital audio at its most fundamental level?',
            'Why is 44100 Hz a standard sample rate?',
        ],
        responses=[
            'Digital audio is a sequence of numbers representing air pressure at regular time intervals. At 44100 Hz, you record 44100 pressure measurements per second. Each measurement (sample) is a number — 16-bit means values from -32768 to 32767. Mixing two sounds is literally adding their number arrays together.',
            '44100 Hz was chosen because human hearing tops out around 20000 Hz, and the Nyquist theorem says you need at least 2x the highest frequency — so 40000 Hz minimum. 44100 was picked for compatibility with early digital recording on video tape (PAL and NTSC frame rates).',
        ],
        bad_responses=['Audio is sound files on computers.'],
    ),
}

print(f'Topic graph: {len(TOPICS)} topics')
for tid, t in TOPICS.items():
    print(f'  [{t.tongue}] {t.name} (depth {t.depth}) -> {len(t.connections)} connections')

In [ ]:
# ============================================================
# Cell 4: Trajectory Pivot Engine
# ============================================================
# Three cognitive flight paths through the same topic graph:
#
#   LINEAR   (baseball throw):
#     direct adjacency-hopping, tight spin, step-by-step
#     coherence demands connected edges; growth rewards small positive steps
#
#   BALLISTIC (football lob):
#     arc UP through deep/abstract topics in first half,
#     then descend back in second half — depth_delta rewarded then reversed
#     coherence is relaxed (0.75/0.40) allowing longer jumps
#
#   CURVE    (soccer kick):
#     rotates through tongue targets in a fixed order (KO→AV→RU→CA→UM→DR→…)
#     coherence comes from hitting the current tongue target, not adjacency
#     spirals back from a completely different language angle
#
# Phi-spiral reinforcement:
#   At Fibonacci-numbered depths (1,2,3,5,8,13…), topics that were already
#   visited get novelty BOOSTED instead of penalised — strategic reinforce.
#
# Sibling diversity:
#   At fork points, siblings get DIFFERENT trajectory modes via _MODE_CYCLE
#   so the same origin fans out through three different cognitive shapes.

from enum import Enum
from dataclasses import dataclass, field
from typing import Optional
from collections import defaultdict
import math, random

class TMode(str, Enum):
    LINEAR    = "linear"
    BALLISTIC = "ballistic"
    CURVE     = "curve"

_TONGUE_ORDER = ["KO", "AV", "RU", "CA", "UM", "DR"]
_FIB = [1, 1]
while _FIB[-1] < 60:
    _FIB.append(_FIB[-1] + _FIB[-2])
FIB_SET = set(_FIB)


@dataclass
class PivotScore:
    relevance: float
    growth: float
    novelty: float
    coherence: float
    trajectory: str = "linear"

    @property
    def quality(self):
        return 0.20*self.relevance + 0.25*self.growth + 0.20*self.novelty + 0.35*self.coherence

    @property
    def is_good(self):
        return self.quality >= 0.5


@dataclass
class TrajState:
    mode: TMode
    step: int = 0
    max_steps: int = 6
    tongue_idx: int = 0
    arc_peaked: bool = False

    @property
    def phase(self):
        return self.step / max(self.max_steps - 1, 1)

    def advance(self):
        return TrajState(self.mode, self.step + 1, self.max_steps, self.tongue_idx, self.arc_peaked)

    def target_tongue(self):
        if self.mode != TMode.CURVE:
            return None
        return _TONGUE_ORDER[self.tongue_idx % len(_TONGUE_ORDER)]

    def next_tongue_cursor(self, used_tongue):
        if self.mode != TMode.CURVE:
            return self.tongue_idx
        return self.tongue_idx + 1 if used_tongue == self.target_tongue() else self.tongue_idx


class TrajectoryPivotEngine:
    _MODE_CYCLE = [TMode.LINEAR, TMode.BALLISTIC, TMode.CURVE]

    def __init__(self, topics, temperature=0.6, branch_threshold=0.68, max_siblings=2,
                 global_visit_decay=0.30, max_branch_budget=500, phi_reinforce=True):
        self.topics = topics
        self.temperature = temperature
        self.branch_threshold = branch_threshold
        self.max_siblings = max_siblings
        self.global_visit_decay = global_visit_decay
        self.max_branch_budget = max_branch_budget
        self.phi_reinforce = phi_reinforce
        self.global_visits = defaultdict(int)
        self.sft_pairs = []
        self.dpo_pairs = []
        self.all_pivot_scores = []
        self.branch_count = 0

    def _score(self, from_id, to_id, local_visits, traj):
        from_t = self.topics[from_id]
        to_t   = self.topics[to_id]
        phase  = traj.phase
        connected = to_id in from_t.connections
        relevance = min(1.0, (1.0 if connected else 0.2) + 0.1 * len(set(from_t.keywords) & set(to_t.keywords)))
        depth_delta = to_t.depth - from_t.depth

        if traj.mode == TMode.LINEAR:
            growth = min(1.0, 0.5 + 0.3 * max(0, depth_delta) - 0.2 * abs(depth_delta))
        elif traj.mode == TMode.BALLISTIC:
            if phase < 0.5:
                growth = min(1.0, 0.3 + 0.6 * max(0, depth_delta))
            else:
                growth = min(1.0, 0.3 + 0.6 * max(0, -depth_delta))
        else:
            target_tongue = traj.target_tongue()
            on_target = to_t.tongue == target_tongue
            tongue_bonus = 0.5 if on_target else (0.2 if to_t.tongue != from_t.tongue else 0.0)
            growth = min(1.0, 0.3 + 0.2 * max(0, depth_delta) + tongue_bonus)

        local_v  = local_visits.get(to_id, 0)
        global_v = self.global_visits.get(to_id, 0)
        novelty  = max(0.0, 1.0 - 0.3 * local_v - self.global_visit_decay * global_v)
        if self.phi_reinforce and traj.step in FIB_SET and global_v > 0:
            novelty = min(1.0, novelty + 0.35)

        if traj.mode == TMode.LINEAR:
            coherence = 0.90 if connected else 0.15
        elif traj.mode == TMode.BALLISTIC:
            coherence = 0.75 if connected else 0.40
        else:
            target_tongue = traj.target_tongue()
            coherence = 0.80 if to_t.tongue == target_tongue else (0.55 if connected else 0.30)

        if from_t.tongue == to_t.tongue and traj.mode != TMode.CURVE:
            coherence = min(1.0, coherence + 0.1)

        return PivotScore(round(relevance,3), round(growth,3), round(novelty,3), round(coherence,3), traj.mode.value)

    def _all_scored(self, current_id, local_visits, traj):
        out = [(tid, self._score(current_id, tid, local_visits, traj)) for tid in self.topics if tid != current_id]
        return sorted(out, key=lambda x: x[1].quality, reverse=True)

    def _softmax_sample(self, candidates, n):
        if not candidates or n <= 0:
            return []
        q = [c[1].quality for c in candidates]
        s = [x / max(self.temperature, 1e-8) for x in q]
        mx = max(s)
        w = [math.exp(x - mx) for x in s]
        tot = sum(w)
        probs = [x / tot for x in w]
        chosen, pool, pp = [], list(range(len(candidates))), list(probs)
        for _ in range(min(n, len(pool))):
            r, cumsum, pick = random.random(), 0.0, pool[-1]
            for i, idx in enumerate(pool):
                cumsum += pp[i]
                if r <= cumsum:
                    pick = idx
                    break
            chosen.append(candidates[pick])
            pos = pool.index(pick)
            pool.pop(pos); pp.pop(pos)
            s2 = sum(pp)
            if s2 > 0:
                pp = [x / s2 for x in pp]
        return chosen

    def _emit(self, topic, turn_i, traj, phi_hit):
        q = topic.questions[turn_i % len(topic.questions)]
        r = topic.responses[turn_i % len(topic.responses)]
        self.sft_pairs.append({
            "instruction": q, "response": r,
            "topic": topic.name, "tongue": topic.tongue, "depth": topic.depth,
            "trajectory": traj.mode.value, "turn": turn_i, "phi_reinforce": phi_hit,
        })
        if topic.bad_responses:
            self.dpo_pairs.append({
                "prompt": q, "chosen": r,
                "rejected": topic.bad_responses[turn_i % len(topic.bad_responses)],
                "topic": topic.name, "tongue": topic.tongue, "trajectory": traj.mode.value,
            })

    def _grow(self, current_id, local_visits, traj, turn_offset, fork_depth):
        if traj.step >= traj.max_steps:
            return
        topic   = self.topics[current_id]
        phi_hit = self.phi_reinforce and traj.step in FIB_SET and self.global_visits[current_id] > 0
        self.global_visits[current_id] += 1
        local_visits = {**local_visits, current_id: local_visits.get(current_id, 0) + 1}
        self._emit(topic, turn_offset + traj.step, traj, phi_hit)
        ranked = self._all_scored(current_id, local_visits, traj)
        if not ranked:
            return
        pick = self._softmax_sample(ranked, 1)
        if not pick:
            return
        next_id, primary_score = pick[0]
        self.all_pivot_scores.append(primary_score)
        next_traj = traj.advance()
        if traj.mode == TMode.CURVE:
            next_traj.tongue_idx = traj.next_tongue_cursor(self.topics[next_id].tongue)
        self._grow(next_id, local_visits, next_traj, turn_offset, fork_depth)
        if primary_score.quality >= self.branch_threshold and self.branch_count < self.max_branch_budget:
            alt_pool = [(tid, s) for tid, s in ranked if tid != next_id]
            siblings = self._softmax_sample(alt_pool, self.max_siblings)
            for i, (sib_id, sib_score) in enumerate(siblings):
                if sib_score.quality >= self.branch_threshold * 0.78:
                    self.branch_count += 1
                    sib_mode = self._MODE_CYCLE[(fork_depth + i + 1) % len(self._MODE_CYCLE)]
                    sib_traj = TrajState(mode=sib_mode, step=traj.step + 1, max_steps=traj.max_steps)
                    self._grow(sib_id, local_visits, sib_traj, turn_offset, fork_depth + 1)

    def generate_tree(self, start_topic, max_steps=6, start_mode=TMode.LINEAR):
        self.branch_count = 0
        sft_b, sc_b = len(self.sft_pairs), len(self.all_pivot_scores)
        traj = TrajState(mode=start_mode, step=0, max_steps=max_steps)
        self._grow(start_topic, {}, traj, turn_offset=0, fork_depth=0)
        new = self.all_pivot_scores[sc_b:]
        avg_q = sum(s.quality for s in new) / max(1, len(new))
        by_tmode = defaultdict(list)
        for s in new:
            by_tmode[s.trajectory].append(s.quality)
        return {
            "start": start_topic, "mode": start_mode.value,
            "sft_added": len(self.sft_pairs) - sft_b,
            "branches_forked": self.branch_count,
            "avg_q": round(avg_q, 4),
            "quality_by_traj": {k: round(sum(v)/len(v), 3) for k, v in by_tmode.items()},
        }


# --- Smoke test ---
random.seed(42)
engine = TrajectoryPivotEngine(TOPICS)
for mode in TMode:
    r = engine.generate_tree("math_binary", max_steps=6, start_mode=mode)
    print(f"  [{mode.value:<10}] sft={r['sft_added']:>4}  forks={r['branches_forked']:>3}  q={r['avg_q']:.4f}  by_traj={r['quality_by_traj']}")


In [ ]:
# ============================================================
# Cell 5: Full Batch — All Topics × All Trajectory Modes + Export
# ============================================================
# Runs one tree per topic × one tree per start mode (LINEAR/BALLISTIC/CURVE).
# Global visits are SHARED so later trees avoid already-covered paths.
# Result: three different cognitive approaches per topic, maximum spread.

def batch_all_trajectories(topics, max_steps=6, temperature=0.6, branch_threshold=0.68,
                           max_siblings=2, seed=None):
    if seed is not None:
        random.seed(seed)
    engine = TrajectoryPivotEngine(
        topics, temperature=temperature, branch_threshold=branch_threshold,
        max_siblings=max_siblings, global_visit_decay=0.30, max_branch_budget=500,
    )
    ordered = sorted(topics.keys(), key=lambda t: topics[t].depth)
    tree_stats = []
    for start in ordered:
        for mode in TMode:
            stat = engine.generate_tree(start, max_steps=max_steps, start_mode=mode)
            tree_stats.append(stat)

    scores = engine.all_pivot_scores
    by_mode = defaultdict(list)
    for s in scores:
        by_mode[s.trajectory].append(s.quality)

    print("=== Batch Complete ===")
    print(f"  SFT pairs : {len(engine.sft_pairs)}")
    print(f"  DPO pairs : {len(engine.dpo_pairs)}")
    print(f"  Pivot evals: {len(scores)}")
    print()
    print("  Quality per trajectory mode:")
    for mode_name, qs in sorted(by_mode.items()):
        avg  = sum(qs) / len(qs)
        good = sum(1 for q in qs if q >= 0.5) / len(qs) * 100
        print(f"    {mode_name:<12} avg={avg:.4f}  good={good:.1f}%  n={len(qs)}")
    print()
    print("  Per-tree breakdown (first 15):")
    for stat in tree_stats[:15]:
        traj_str = " ".join(f"{k}={v:.3f}" for k,v in stat['quality_by_traj'].items())
        print(f"    [{stat['start']:<22} {stat['mode']:<10}]  sft={stat['sft_added']:>4}  "
              f"forks={stat['branches_forked']:>3}  q={stat['avg_q']:.3f}  {traj_str}")
    print()
    print("  Global visit spread:")
    for tid, count in sorted(engine.global_visits.items(), key=lambda x: -x[1]):
        print(f"    {tid:<25} {count:>3}  {'#' * min(count, 40)}")

    return engine.sft_pairs, engine.dpo_pairs, tree_stats


sft_data, dpo_data, tree_stats = batch_all_trajectories(
    TOPICS, max_steps=6, temperature=0.6, branch_threshold=0.68, max_siblings=2, seed=42,
)

# Save to Drive
output_dir = Path('/content/drive/MyDrive/SCBE_Training')
output_dir.mkdir(parents=True, exist_ok=True)

with open(output_dir / 'pivot_sft_v3.jsonl', 'w') as f:
    for row in sft_data:
        f.write(json.dumps(row) + '\n')

with open(output_dir / 'pivot_dpo_v3.jsonl', 'w') as f:
    for row in dpo_data:
        f.write(json.dumps(row) + '\n')

print(f'\nSaved to {output_dir}')
print(f'  pivot_sft_v3.jsonl  ({len(sft_data)} rows)')
print(f'  pivot_dpo_v3.jsonl  ({len(dpo_data)} rows)')


In [ ]:
# ============================================================
# Cell 6: Push to HuggingFace
# ============================================================

from huggingface_hub import HfApi
from google.colab import userdata

# Set your HF token in Colab secrets (key: HF_TOKEN)
try:
    hf_token = userdata.get('HF_TOKEN')
except:
    hf_token = input('Enter HuggingFace token: ')

api = HfApi(token=hf_token)

DATASET_REPO = 'issdandavis/scbe-aethermoore-training-data'

# Upload SFT
api.upload_file(
    path_or_fileobj=str(output_dir / 'pivot_sft_v2.jsonl'),
    path_in_repo='pivot/pivot_sft_v2.jsonl',
    repo_id=DATASET_REPO,
    repo_type='dataset',
)

# Upload DPO
api.upload_file(
    path_or_fileobj=str(output_dir / 'pivot_dpo_v2.jsonl'),
    path_in_repo='pivot/pivot_dpo_v2.jsonl',
    repo_id=DATASET_REPO,
    repo_type='dataset',
)

print(f'Pushed to https://huggingface.co/datasets/{DATASET_REPO}')
print('  pivot/pivot_sft_v2.jsonl')
print('  pivot/pivot_dpo_v2.jsonl')

## Next Steps

1. **Expand topics** — Add math textbook content (algebra, trig, calculus, linear algebra, hyperbolic geometry)
2. **Add book chapters** — Parse The Six Tongues Protocol for character dialogue SFT pairs
3. **Add Everweave logs** — Convert game paragraphs to instruction-response format
4. **ChoiceScript converter** — Parse .txt choice files into DPO preference pairs
5. **Run training** — Use Unsloth on Colab T4 to fine-tune with combined SFT + DPO
6. **Measure** — Track pivot quality metrics across training runs